# 02 — SigProfiler matrix generation and signature fitting

This notebook:
- generates the SBS96 matrix from the MAF input,
- fits the matrix to COSMIC signatures,
- previews the fitted activity table,
- prints one representative fitted sample with reconstruction quality.


In [8]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from SigProfilerAssignment import Analyzer as Analyze
from SigProfilerMatrixGenerator.scripts import SigProfilerMatrixGeneratorFunc

PROJECT_ROOT = Path.cwd().resolve()

pd.set_option("display.max_columns", 120)

def get_sbs_columns(columns):
    sbs_columns = [col for col in columns if str(col).startswith("SBS") and str(col)[3:].isdigit()]
    return sorted(sbs_columns, key=lambda col: int(str(col)[3:]))

## 1. Define input and output paths


In [9]:
maf_input_dir = PROJECT_ROOT / "data" / "maf" / "input"
matrix_output_dir = PROJECT_ROOT / "data" / "maf" / "output"
signature_fit_dir = PROJECT_ROOT / "results" / "LUAD_sig_output"

signature_fit_dir.mkdir(parents=True, exist_ok=True)

pd.DataFrame({
    "Path": ["MAF input", "Matrix output", "Signature fit output"],
    "Location": [maf_input_dir, matrix_output_dir, signature_fit_dir],
})

,Path,Location
0,MAF input,/Users/michaljendrusak/PycharmProjects/tcga-lu...
1,Matrix output,/Users/michaljendrusak/PycharmProjects/tcga-lu...
2,Signature fit output,/Users/michaljendrusak/PycharmProjects/tcga-lu...


## 2. Preview the MAF input folder


In [10]:
files = sorted(
    path for path in maf_input_dir.rglob("*")
    if path.is_file() and path.suffix.lower() in {".maf", ".tsv", ".txt"}
)

pd.DataFrame({
    "file_name": [f.name for f in files],
    "folder": [f.parent.name for f in files],
}).head(10)

,file_name,folder
0,00c2e7e1-5cac-4ff6-84ac-68d01caf2bd3.wxs.aliqu...,input
1,0196bd59-8bff-46da-a06a-4114fb1ffdb6.wxs.aliqu...,input
2,01e8d53d-3dc4-4fde-9fe2-f8586469628a.wxs.aliqu...,input
3,02b85206-3f54-4b9d-8a11-e0d5ddaad387.wxs.aliqu...,input
4,0307f560-777b-4492-bffd-8eacae8da208.wxs.aliqu...,input
5,037ecdc2-dc6a-45fd-8eb3-6fa1fc4db46e.wxs.aliqu...,input
6,04b06599-0300-4e8e-bffb-72cf5825791d.wxs.aliqu...,input
7,061b49bc-0a5e-4801-bfaf-88ca7231feef.wxs.aliqu...,input
8,074862f2-acbf-4141-a72a-da28c9cb5f39.wxs.aliqu...,input
9,07bbadef-2815-4742-a5d4-ec8af2b9610f.wxs.aliqu...,input


## 3. Generate the SBS96 matrix


In [4]:
matrices = SigProfilerMatrixGeneratorFunc.SigProfilerMatrixGeneratorFunc(
    project="LUAD",
    reference_genome="GRCh38",
    path_to_input_files=str(maf_input_dir),
    plot=False,
)

print("SBS96 matrix generation finished.")


Starting matrix generation for SNVs and DINUCs...Completed! Elapsed time: 20.75 seconds.
Starting matrix generation for INDELs...Completed! Elapsed time: 6.68 seconds.
Matrices generated for 616 samples with 108 errors. Total of 188298 SNVs, 2876 DINUCs, and 6323 INDELs were successfully analyzed.
SBS96 matrix generation finished.


## 4. Load and preview the generated matrix


In [5]:
sbs96_path = matrix_output_dir / "SBS" / "LUAD.SBS96.all"

matrix_df = pd.read_csv(sbs96_path, sep="\t")

display(
    pd.DataFrame({
        "file": [str(sbs96_path.relative_to(PROJECT_ROOT))],
        "rows": [matrix_df.shape[0]],
        "columns": [matrix_df.shape[1]],
    })
)

display(matrix_df.iloc[:5, :8])

,file,rows,columns
0,data/maf/output/SBS/LUAD.SBS96.all,96,617


,MutationType,TCGA-05-4244-01A-01D-1105-08,TCGA-05-4249-01A-01D-1105-08,TCGA-05-4250-01A-01D-1105-08,TCGA-05-4382-01A-01D-1931-08,TCGA-05-4384-01A-01D-1753-08,TCGA-05-4389-01A-01D-1265-08,TCGA-05-4390-01A-02D-1753-08
0,A[C>A]A,5,8,7,50,5,1,16
1,A[C>A]C,5,8,1,62,1,2,10
2,A[C>A]G,4,4,9,23,3,0,14
3,A[C>A]T,2,6,11,40,1,2,9
4,A[C>G]A,3,3,0,4,0,0,0


## 5. Fit the SBS96 matrix to COSMIC signatures


In [6]:
Analyze.cosmic_fit(
    samples=str(sbs96_path),
    output=str(signature_fit_dir),
    input_type="matrix",
    context_type="96",
    collapse_to_SBS96=True,
    cosmic_version=3.4,
)

display(
    pd.DataFrame({
        "step": ["COSMIC fitting finished"],
        "input_file": [str(sbs96_path.relative_to(PROJECT_ROOT))],
        "output_folder": [str(signature_fit_dir.relative_to(PROJECT_ROOT))],
        "context_type": ["96"],
        "cosmic_version": [3.4],
    })
)

Assigning COSMIC sigs or Signature Database ...... 


[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done   2 tasks      | elapsed:    5.5s
[Parallel(n_jobs=8)]: Done  56 tasks      | elapsed:   10.3s
[Parallel(n_jobs=8)]: Done 146 tasks      | elapsed:   18.5s
[Parallel(n_jobs=8)]: Done 272 tasks      | elapsed:   33.6s
[Parallel(n_jobs=8)]: Done 434 tasks      | elapsed:   53.6s
[Parallel(n_jobs=8)]: Done 616 out of 616 | elapsed:  1.2min finished




 
Your Job Is Successfully Completed! Thank You For Using SigProfilerAssignment.
 


,step,input_file,output_folder,context_type,cosmic_version
0,COSMIC fitting finished,data/maf/output/SBS/LUAD.SBS96.all,results/LUAD_sig_output,96,3.4


## 6. Load the fitted activity table


In [11]:
activities_file = (
    signature_fit_dir
    / "Assignment_Solution"
    / "Activities"
    / "Assignment_Solution_Activities.txt"
)

activities = pd.read_csv(activities_file, sep="\t", index_col=0)

signature_cols = get_sbs_columns(activities.columns)
activity_matrix = activities[signature_cols].copy()

display(
    pd.DataFrame({
        "table": ["Activities"],
        "samples": [activity_matrix.shape[0]],
        "fitted_signatures": [len(signature_cols)],
    })
)

display(activity_matrix.iloc[:5, :10])

non_zero_signature_totals = (
    activity_matrix.sum(axis=0)
    .loc[lambda s: s > 0]
    .sort_values(ascending=False)
    .rename("total_activity")
    .reset_index()
    .rename(columns={"index": "signature"})
)

display(non_zero_signature_totals.head(10))

,table,samples,fitted_signatures
0,Activities,616,71


,SBS1,SBS2,SBS3,SBS4,SBS5,SBS6,SBS8,SBS9,SBS11,SBS12
Samples,,,,,,,,,,
TCGA-05-4244-01A-01D-1105-08,4,0,0,150,47,19,0,0,0,0
TCGA-05-4249-01A-01D-1105-08,7,0,0,201,86,0,0,0,0,0
TCGA-05-4250-01A-01D-1105-08,9,21,0,207,73,0,0,0,0,0
TCGA-05-4382-01A-01D-1931-08,45,97,0,1508,65,0,0,0,0,0
TCGA-05-4384-01A-01D-1753-08,0,0,0,78,44,0,0,0,0,0


,signature,total_activity
0,SBS4,109113
1,SBS5,34770
2,SBS13,8958
3,SBS2,8412
4,SBS1,6386
5,SBS24,3645
6,SBS39,1391
7,SBS15,1236
8,SBS6,1090
9,SBS87,986


## 7. Print one representative fitted composition


In [12]:
stats_file = next((signature_fit_dir / "Assignment_Solution").rglob("*Stats*.txt"))
stats_df = pd.read_csv(stats_file, sep="\t")

sample_col = [c for c in stats_df.columns if "sample" in c.lower()][0]
cosine_col = [c for c in stats_df.columns if "cosine" in c.lower()][0]
l2_col = [c for c in stats_df.columns if "l2" in c.lower()][0]

best_row = stats_df.sort_values(cosine_col, ascending=False).iloc[0]
best_sample = best_row[sample_col]

active_signatures = activity_matrix.loc[best_sample]
active_signatures = active_signatures[active_signatures > 0].sort_values(ascending=False).reset_index()
active_signatures.columns = ["signature", "activity"]

display(pd.DataFrame({
    "sample": [best_sample],
    "cosine_similarity": [best_row[cosine_col]],
    "l2_error": [best_row[l2_col]],
}))

display(active_signatures)

,sample,cosine_similarity,l2_error
0,TCGA-55-A4DF-01A-11D-A24D-08,0.986,42.665


,signature,activity
0,SBS13,290
1,SBS2,206
2,SBS4,192
3,SBS1,41
